# RAG with Unified Lineage

Demonstrates the RudriQ thesis: a single import line gives you a unified trace from raw data through LLM call.

**v0.0.1 status:** This notebook shows the *intended* architecture. Linking is stubbed; it activates in v0.1.

## 1. Activate the unified trace

In [ ]:
import rudriq.auto    # one import: lineage + LLM + linker

## 2. A trivial RAG pipeline

Read documents, embed, retrieve, generate. RudriQ traces every step.

In [ ]:
import pandas as pd

# AutoLineage tracks this read
docs = pd.read_csv('docs.csv')

# AutoLineage tracks this filter
docs = docs[docs['lang'] == 'en']

# OpenLLMetry tracks the embeddings call below
# RudriQ links the embedding's input back to the filtered DataFrame above
# from openai import OpenAI
# client = OpenAI()
# embeddings = client.embeddings.create(
#     model='text-embedding-3-small',
#     input=docs['text'].tolist(),
# )

print(f'Pipeline ran on {len(docs)} documents.')

## 3. Inspect the unified trace

In [ ]:
from rudriq import get_tracker

tracker = get_tracker()
graph = tracker.get_full_graph()
print(f"Captured {graph['node_count']} operations across data and LLM domains.")

## 4. Diagnose root cause when something changes

When response quality drops, RudriQ walks the unified trace to find the upstream operation responsible. v0.2 ships full scoring; v0.0.1 returns a structural summary.

In [ ]:
from rudriq import diagnose

result = diagnose(target_metric='answer_quality')
print(result)

## 5. Generate an audit report

Same trace, different output. The audit report enumerates every data operation, every LLM call, and every cross-domain link in a structured form.

In [ ]:
from rudriq.export.audit import generate_audit_report

report = generate_audit_report()
print(f"Data operations:     {len(report['data_operations'])}")
print(f"LLM operations:      {len(report['llm_operations'])}")
print(f"Cross-domain links:  {len(report['cross_domain_links'])}")